#### Librerias

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import re
import random
import joblib
import nltk
from nltk.corpus import stopwords

Mounted at /content/drive


#### Variables

In [3]:
MODELOS_PATH = "/content/drive/MyDrive/Trabajo práctico 3/modelos"

#### Carga de modelo y vectorizador (LR optimizado)

In [4]:
modelo = joblib.load(f"{MODELOS_PATH}/modelo_lr_optimizado.pkl")
vectorizer = joblib.load(f"{MODELOS_PATH}/vectorizer_tfidf.pkl")

#### Función de limpieza

Se repite acá porque cada notebook de Colab es independiente.

In [5]:
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

def limpieza(texto):
    texto = texto.lower()
    texto = re.sub(r'http\S+|www\S+', '', texto)
    texto = re.sub(r'@\w+', '', texto)
    texto = re.sub(r'#', '', texto)
    texto = re.sub(r'[^a-z\s]', '', texto)
    texto = re.sub(r'\s+', ' ', texto).strip()
    tokens = [t for t in texto.split() if t not in stop_words]
    return ' '.join(tokens)

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


#### Preguntas de la cita

In [6]:
preguntas = [
    "¿Cómo me ves hoy? ¿Estoy linda?",
    "¿Cuál es lo más vergonzoso que te pasó en una cita?",
    "¿Alguna vez te hicieron ghosting? ¿Cómo te sentiste?",
    "¿Qué diría tu ex sobre vos?",
    "Describí tu peor primera cita de la historia.",
    "¿Qué hábito tuyo podría molestarle a una pareja?",
    "¿Cómo te sentís cuando te preguntan 'a dónde va esto'?",
    "¿Cuál es tu mayor miedo con las citas en este momento?",
]

#### Clasificación con umbral de confianza

In [7]:
!pip install deep-translator -q
from deep_translator import GoogleTranslator

def clasificar_respuesta(texto, umbral=0.60):
    texto_en = GoogleTranslator(source='es', target='en').translate(texto)
    texto_limpio = limpieza(texto_en)
    vector = vectorizer.transform([texto_limpio])
    proba = modelo.predict_proba(vector)[0]

    if max(proba) < umbral:
        return "Dudoso 😐", proba, texto_en
    elif proba[1] > proba[0]:
        return "Positivo 😊", proba, texto_en
    else:
        return "Negativo 😡", proba, texto_en

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 3.6 MB/s eta 0:00:00


#### El juego

In [8]:
def jugar_cita(n_rondas=5):
    print("Bienvenido/a a 'Speed Dating con IA' 💘")
    print("Te voy a hacer algunas preguntas incómodas. Contestá con sinceridad...\n")
    preguntas_ronda = random.sample(preguntas, n_rondas)
    for pregunta in preguntas_ronda:
        print(f"\n💬 {pregunta}")
        respuesta = input("Tu respuesta: ")
        resultado, proba, texto_en = clasificar_respuesta(respuesta)
        print(f"El modelo lee tu vibra como: {resultado} (confianza: {max(proba)*100:.1f}%)")
    print("\n¡Gracias por la cita! 😉")

#### Correr el juego

In [9]:
jugar_cita()

Bienvenido/a a 'Speed Dating con IA' 💘
Te voy a hacer algunas preguntas incómodas. Contestá con sinceridad...


💬 ¿Cuál es tu mayor miedo con las citas en este momento?
Tu respuesta: Que no me des un beso
El modelo lee tu vibra como: Dudoso 😐 (confianza: 51.9%)

💬 ¿Qué hábito tuyo podría molestarle a una pareja?
Tu respuesta: Me gusta engañarla de vez en cuando
El modelo lee tu vibra como: Positivo 😊 (confianza: 69.2%)

💬 ¿Cómo me ves hoy? ¿Estoy linda?
Tu respuesta: No, estas fea
El modelo lee tu vibra como: Negativo 😡 (confianza: 64.9%)

💬 ¿Cómo te sentís cuando te preguntan 'a dónde va esto'?
Tu respuesta: inocmodo
El modelo lee tu vibra como: Negativo 😡 (confianza: 89.8%)

💬 ¿Cuál es lo más vergonzoso que te pasó en una cita?
Tu respuesta: Me dijo que tenia un lunar muy feo y despues me pego
El modelo lee tu vibra como: Negativo 😡 (confianza: 75.4%)

¡Gracias por la cita! 😉
